# Prophet — All Forces 2026 Full-Year Forecast (Jan–Dec)

Mirrors notebook 10 (TimesFM all-forces) using **Facebook Prophet** with UK public holidays.  
Forecasts Jan–Dec 2026 at force level for all 43 forces.  
Evaluates against Jan–Mar 2026 actuals. Apr–Dec shown as forward forecast only.

**Training context:** 2012–2019 + 2022–2025 (pandemic years excluded)  
**Crime types:** all 16  
**Evaluation window:** Jan–Mar 2026  
**Forecast window:** Jan–Dec 2026

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
from prophet import Prophet
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

## 1. Load and aggregate to force level

In [ ]:
CRIME_TYPES = [
    'Violence and sexual offences', 'Criminal damage and arson', 'Drugs',
    'Violent crime', 'Burglary', 'Other theft', 'Vehicle crime',
    'Public order', 'Other crime', 'Shoplifting', 'Robbery',
    'Bicycle theft', 'Theft from the person', 'Possession of weapons',
    'Public disorder and weapons', 'Anti-social behaviour',
]
EXCLUDE_FORCES = {
    'British Transport Police',          # national rail force, funded separately
    'Police Service of Northern Ireland', # different jurisdiction
}
TRAIN_YEARS     = set(range(2012, 2020)) | set(range(2022, 2026))
FORECAST_MONTHS = pd.date_range('2026-01-01', periods=12, freq='MS')  # Jan–Dec 2026
EVAL_MONTHS     = pd.date_range('2026-01-01', periods=3, freq='MS')   # Jan–Mar (actuals available)

raw = pd.read_parquet('../data/processed/crimes_clean_dedup_all_years.parquet')
raw = raw[raw['Crime type'].isin(CRIME_TYPES) & ~raw['Falls within'].isin(EXCLUDE_FORCES)].copy()
raw['month'] = pd.to_datetime(raw['Month'])
raw = raw[raw['month'].dt.year >= 2012]

force_monthly = (
    raw.groupby(['Falls within', 'month'])
    .size()
    .reset_index(name='count')
    .rename(columns={'Falls within': 'force'})
)

all_forces = sorted(force_monthly['force'].unique())
print(f'Forces: {len(all_forces)}')
print(f'Excluded: {EXCLUDE_FORCES}')
print(f'Date range: {force_monthly["month"].min().date()} → {force_monthly["month"].max().date()}')

## 2. Prophet model

Parameters:
- `seasonality_mode='multiplicative'` — crime scales with level
- `yearly_seasonality=True` — strong annual patterns
- `add_country_holidays('GB')` — UK bank holidays
- `changepoint_prior_scale=0.15` — moderate trend flexibility
- `seasonality_prior_scale=10` — strong seasonality allowed

Note: pandemic gap (2020–2021) is handled naturally — Prophet treats it as missing data and fits trend/seasonality around it.

In [ ]:
def prophet_forecast(train_series, n_months):
    df = train_series.reset_index()
    df.columns = ['ds', 'y']
    df['ds'] = pd.to_datetime(df['ds'])

    m = Prophet(
        seasonality_mode='multiplicative',
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.15,
        seasonality_prior_scale=10,
        uncertainty_samples=0,
    )
    m.add_country_holidays(country_name='GB')
    m.fit(df)

    future = m.make_future_dataframe(periods=n_months, freq='MS', include_history=False)
    forecast = m.predict(future)
    return np.clip(forecast['yhat'].values[:n_months], 0, None)

## 3. Fit and forecast all forces

In [ ]:
forecast_rows = []

for i, force in enumerate(all_forces, 1):
    print(f'[{i:02d}/{len(all_forces)}] {force}', end='  ')

    train = (
        force_monthly[
            (force_monthly['force'] == force) &
            (force_monthly['month'].dt.year.isin(TRAIN_YEARS))
        ]
        .set_index('month')['count']
        .sort_index()
    )

    if len(train) < 12:
        print('SKIPPED (insufficient history)')
        continue

    preds = prophet_forecast(train, 12)
    print(f'Jan={preds[0]:.0f}  Feb={preds[1]:.0f}  Mar={preds[2]:.0f}  '
          f'Apr={preds[3]:.0f}  May={preds[4]:.0f}  Jun={preds[5]:.0f}  '
          f'Jul={preds[6]:.0f}  Aug={preds[7]:.0f}  Sep={preds[8]:.0f}  '
          f'Oct={preds[9]:.0f}  Nov={preds[10]:.0f}  Dec={preds[11]:.0f}')

    for j, month in enumerate(FORECAST_MONTHS):
        forecast_rows.append({
            'force': force,
            'month': month,
            'forecast': preds[j],
            'has_actual': month in EVAL_MONTHS,
        })

forecast_df = pd.DataFrame(forecast_rows)
print(f'\nTotal forecast rows: {len(forecast_df):,}')

## 4. Attach actuals and baseline

In [ ]:
actuals_26 = force_monthly[force_monthly['month'].isin(EVAL_MONTHS)].copy()
forecast_df = forecast_df.merge(
    actuals_26[['force', 'month', 'count']].rename(columns={'count': 'actual'}),
    on=['force', 'month'], how='left'
)

baseline = (
    force_monthly[force_monthly['month'].dt.year == 2025]
    .groupby('force')['count'].mean()
    .rename('baseline').reset_index()
)
forecast_df = forecast_df.merge(baseline, on='force', how='left')

# Load TimesFM all-forces results for comparison if available
try:
    tfm_eval = pd.read_csv('../outputs/timesfm_all_forces_q1_eval.csv')
    has_tfm = True
    print('TimesFM results loaded for comparison.')
except FileNotFoundError:
    has_tfm = False
    print('TimesFM results not found — run notebook 10 first for comparison.')

## 5. Evaluation metrics (Jan–Mar 2026)

In [ ]:
# metric helpers (defined once, reused across the notebook)
def mae(pred, actual):
    return float(np.mean(np.abs(pred - actual)))

def rmse(pred, actual):
    return float(np.sqrt(np.mean((pred - actual) ** 2)))

def mape(pred, actual):
    # guard against division by zero where actual demand is 0
    return float(np.mean(np.abs((pred - actual) / np.where(actual == 0, 1, actual))) * 100)

eval_rows = []
eval_data = forecast_df[forecast_df['actual'].notna()]

for force, g in eval_data.groupby('force'):
    actual_vals   = g['actual'].values.astype(float)
    forecast_vals = g['forecast'].values
    baseline_vals = g['baseline'].values

    eval_rows.append({
        'force':          force,
        'baseline_mae':   mae(baseline_vals, actual_vals),
        'model_mae':      mae(forecast_vals,  actual_vals),
        'baseline_rmse':  rmse(baseline_vals, actual_vals),
        'model_rmse':     rmse(forecast_vals,  actual_vals),
        'model_mape':     mape(forecast_vals,  actual_vals),
        'mean_actual':    actual_vals.mean(),
        'mean_forecast':  forecast_vals.mean(),
    })

eval_df = pd.DataFrame(eval_rows)
eval_df['delta_mae']  = eval_df['baseline_mae'] - eval_df['model_mae']
eval_df['rmae']       = eval_df['model_mae'] / eval_df['baseline_mae']
eval_df['model_wins'] = eval_df['model_mae'] < eval_df['baseline_mae']
eval_df = eval_df.sort_values('model_mape').reset_index(drop=True)

print(f'Overall — MAE: {eval_df["model_mae"].mean():.1f}  '
      f'MAPE: {eval_df["model_mape"].mean():.1f}%  '
      f'Win rate: {eval_df["model_wins"].mean()*100:.0f}%  '
      f'RMAE: {eval_df["rmae"].mean():.3f}')
print()
display(
    eval_df[['force','mean_actual','mean_forecast','baseline_mae','model_mae',
             'delta_mae','rmae','model_mape','model_wins']]
    .round(1)
    .rename(columns={
        'force':'Force','mean_actual':'Avg Actual','mean_forecast':'Avg Forecast',
        'baseline_mae':'Baseline MAE','model_mae':'Prophet MAE',
        'delta_mae':'Δ MAE','rmae':'RMAE','model_mape':'MAPE %','model_wins':'Wins'
    })
    .set_index('Force')
)

## 6. Prophet vs TimesFM vs Baseline comparison

In [ ]:
if has_tfm:
    comp = eval_df[['force','baseline_mae','model_mae','rmae','model_mape','model_wins']].rename(
        columns={'model_mae':'prophet_mae','rmae':'prophet_rmae',
                 'model_mape':'prophet_mape','model_wins':'prophet_wins'}
    ).merge(
        tfm_eval[['force','model_mae','rmae','model_mape','model_wins']].rename(
            columns={'model_mae':'tfm_mae','rmae':'tfm_rmae',
                     'model_mape':'tfm_mape','model_wins':'tfm_wins'}
        ),
        on='force', how='inner'
    ).sort_values('prophet_mape')

    comp['better_model'] = np.where(
        comp['prophet_mape'] < comp['tfm_mape'], 'Prophet', 'TimesFM'
    )

    print(f"Prophet wins:  {(comp['better_model']=='Prophet').sum()}/{len(comp)} forces")
    print(f"TimesFM wins:  {(comp['better_model']=='TimesFM').sum()}/{len(comp)} forces\n")

    display(
        comp[['force','baseline_mae','prophet_mae','prophet_mape',
              'tfm_mae','tfm_mape','better_model']]
        .round(1)
        .rename(columns={
            'force':'Force','baseline_mae':'Baseline MAE',
            'prophet_mae':'Prophet MAE','prophet_mape':'Prophet MAPE%',
            'tfm_mae':'TimesFM MAE','tfm_mape':'TFM MAPE%','better_model':'Winner'
        })
        .set_index('Force')
    )
else:
    print('Run notebook 10 first to enable Prophet vs TimesFM comparison.')

## 7. MAPE ranking chart

In [ ]:
if has_tfm:
    # Side-by-side MAPE: Prophet vs TimesFM
    merged = eval_df[['force','model_mape']].rename(columns={'model_mape':'prophet_mape'}).merge(
        tfm_eval[['force','model_mape']].rename(columns={'model_mape':'tfm_mape'}),
        on='force'
    ).sort_values('prophet_mape')

    fig, ax = plt.subplots(figsize=(10, 13))
    y = np.arange(len(merged))
    w = 0.38
    ax.barh(y - w/2, merged['prophet_mape'], w, label='Prophet',  color='#4CAF50', alpha=0.85)
    ax.barh(y + w/2, merged['tfm_mape'],     w, label='TimesFM', color='#2196F3', alpha=0.85)
    ax.set_yticks(y)
    ax.set_yticklabels(
        merged['force'].str.replace(' Constabulary','').str.replace(' Police','').str.replace(' Service',''),
        fontsize=8
    )
    ax.axvline(10, color='orange', linestyle='--', linewidth=1, label='10%')
    ax.axvline(20, color='red',    linestyle='--', linewidth=1, label='20%')
    ax.set_xlabel('MAPE % (lower = better)')
    ax.set_title('Prophet vs TimesFM — MAPE % by Force\n(Jan–Mar 2026 evaluation)', fontweight='bold')
    ax.legend(loc='lower right')
    plt.tight_layout()
    os.makedirs('../outputs', exist_ok=True)
    plt.savefig('../outputs/prophet_vs_timesfm_mape_all_forces.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    # Prophet only
    fig, ax = plt.subplots(figsize=(10, 12))
    colors = ['#4CAF50' if w else '#F44336' for w in eval_df['model_wins']]
    bars = ax.barh(
        eval_df['force'].str.replace(' Constabulary','').str.replace(' Police','').str.replace(' Service',''),
        eval_df['model_mape'], color=colors, edgecolor='white', linewidth=0.5
    )
    ax.axvline(10, color='orange', linestyle='--', linewidth=1, label='10%')
    ax.axvline(20, color='red',    linestyle='--', linewidth=1, label='20%')
    for bar, v in zip(bars, eval_df['model_mape']):
        ax.text(v + 0.2, bar.get_y() + bar.get_height()/2, f'{v:.1f}%', va='center', fontsize=7)
    ax.set_xlabel('MAPE %')
    ax.set_title('Prophet MAPE % by Force — Jan–Mar 2026\n(green = beats baseline)', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    os.makedirs('../outputs', exist_ok=True)
    plt.savefig('../outputs/prophet_mape_all_forces.png', dpi=150, bbox_inches='tight')
    plt.show()

## 8. Forecast vs Actual grid (all forces)

In [ ]:
HISTORY_START = '2023-01-01'
ncols = 5
nrows = int(np.ceil(len(all_forces) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(22, nrows * 3.5))
axes_flat = axes.flatten()
fig.suptitle('Prophet 2026 Forecast (Jan–Jun) vs Actuals (Jan–Mar) — All Forces',
             fontsize=13, fontweight='bold')

for i, force in enumerate(all_forces):
    ax = axes_flat[i]

    history = force_monthly[
        (force_monthly['force'] == force) &
        (force_monthly['month'] >= HISTORY_START) &
        (force_monthly['month'].dt.year.isin(TRAIN_YEARS))
    ].sort_values('month')
    ax.plot(history['month'], history['count'], color='#94a3b8', linewidth=1, label='History')

    fc = forecast_df[forecast_df['force'] == force].sort_values('month')
    if fc.empty:
        ax.set_title(force[:20], fontsize=7)
        continue

    ax.plot(fc['month'], fc['forecast'], color='#4CAF50', linewidth=1.8,
            marker='s', markersize=4, linestyle='--', label='Prophet', zorder=5)

    fc_eval = fc[fc['actual'].notna()]
    if not fc_eval.empty:
        ax.plot(fc_eval['month'], fc_eval['actual'], color='#FF5722', linewidth=2,
                marker='o', markersize=5, label='Actual', zorder=6)

    # Vertical split: evaluated | forecast only
    split = pd.Timestamp('2026-04-01')
    ax.axvline(split, color='#9E9E9E', linestyle=':', linewidth=1)
    ax.axvspan(split, FORECAST_MONTHS[-1] + pd.DateOffset(months=1), alpha=0.06, color='green')

    bl = baseline[baseline['force'] == force]['baseline'].values
    if len(bl):
        ax.axhline(bl[0], color='#9E9E9E', linestyle=':', linewidth=0.8)

    ax.set_title(
        force.replace(' Constabulary','').replace(' Police','').replace(' Service',''),
        fontsize=8, fontweight='bold'
    )
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b%y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.tick_params(axis='x', labelsize=6, rotation=30)
    ax.tick_params(axis='y', labelsize=6)

    mape_row = eval_df[eval_df['force'] == force]
    if not mape_row.empty:
        mape_val = mape_row['model_mape'].values[0]
        wins     = mape_row['model_wins'].values[0]
        ax.text(0.98, 0.97, f'{mape_val:.1f}%', transform=ax.transAxes,
                ha='right', va='top', fontsize=7, fontweight='bold',
                color='#4CAF50' if wins else '#F44336')

for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig('../outputs/prophet_all_forces_forecast_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. H1 2026 forecast totals — budget allocation view

In [ ]:
full_year_forecast = (
    forecast_df.groupby('force')['forecast'].sum()
    .reset_index().rename(columns={'forecast': 'forecast_total'})
)
q1_actual = (
    forecast_df[forecast_df['actual'].notna()]
    .groupby('force')['actual'].sum()
    .reset_index().rename(columns={'actual': 'q1_actual_total'})
)
full_2025 = (
    force_monthly[force_monthly['month'].dt.year == 2025]
    .groupby('force')['count'].sum()
    .reset_index().rename(columns={'count': 'full_2025_actual'})
)

budget_view = (
    full_year_forecast
    .merge(q1_actual, on='force', how='left')
    .merge(full_2025, on='force', how='left')
    .merge(eval_df[['force','model_mape','model_wins']], on='force', how='left')
)
budget_view['demand_share_pct'] = (
    budget_view['forecast_total'] / budget_view['forecast_total'].sum() * 100
).round(2)
budget_view['yoy_change_pct'] = (
    (budget_view['forecast_total'] - budget_view['full_2025_actual']) /
    budget_view['full_2025_actual'] * 100
).round(1)

# Merge TimesFM budget view for comparison
try:
    tfm_budget = pd.read_csv('../outputs/timesfm_all_forces_budget_view.csv')
    budget_view = budget_view.merge(
        tfm_budget[['force','forecast_total','demand_share_pct']].rename(
            columns={'forecast_total':'tfm_forecast_total','demand_share_pct':'tfm_demand_share_pct'}
        ),
        on='force', how='left'
    )
    has_tfm_budget = True
except FileNotFoundError:
    has_tfm_budget = False

budget_view = budget_view.sort_values('forecast_total', ascending=False).reset_index(drop=True)

display_cols = ['force','forecast_total','q1_actual_total','full_2025_actual',
                'demand_share_pct','yoy_change_pct','model_mape']
if has_tfm_budget:
    display_cols += ['tfm_forecast_total','tfm_demand_share_pct']

print('=== Full-Year 2026 Forecast — Budget Allocation View ===')
display(
    budget_view[display_cols].round(1)
    .rename(columns={
        'force':'Force','forecast_total':'Prophet 2026 Forecast',
        'q1_actual_total':'Q1 Actual','full_2025_actual':'2025 Actual (Full Year)',
        'demand_share_pct':'Prophet Share %','yoy_change_pct':'YoY %',
        'model_mape':'MAPE %','tfm_forecast_total':'TFM 2026 Forecast',
        'tfm_demand_share_pct':'TFM Share %',
    })
    .set_index('Force')
)

In [ ]:
top_n = budget_view.head(20)
force_labels_short = (
    top_n['force']
    .str.replace(' Constabulary','').str.replace(' Police','')
    .str.replace(' Service','')
)

if has_tfm_budget:
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))

fig.suptitle('Full-Year 2026 Demand Forecast — Prophet (All Forces)', fontsize=13, fontweight='bold')

# Prophet forecast totals (top 20)
axes[0].barh(force_labels_short, top_n['forecast_total'],
             color='#4CAF50', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Total forecast incidents (2026)')
axes[0].set_title('Top 20 Forces by Prophet Forecast Demand', fontsize=10)
axes[0].invert_yaxis()
axes[0].tick_params(axis='y', labelsize=8)

# YoY change
yoy = budget_view.dropna(subset=['yoy_change_pct']).sort_values('yoy_change_pct')
yoy_short = yoy['force'].str.replace(' Constabulary','').str.replace(' Police','').str.replace(' Service','')
yoy_colors = ['#4CAF50' if v <= 0 else '#F44336' for v in yoy['yoy_change_pct']]
axes[1].barh(yoy_short, yoy['yoy_change_pct'], color=yoy_colors, edgecolor='white', linewidth=0.5)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('YoY % vs 2025')
axes[1].set_title('Forecast YoY Change vs 2025\n(green = lower, red = higher demand)', fontsize=10)
axes[1].tick_params(axis='y', labelsize=7)

# Prophet vs TimesFM demand share comparison
if has_tfm_budget:
    share_df = budget_view.dropna(subset=['tfm_demand_share_pct']).sort_values('demand_share_pct', ascending=False).head(20)
    share_short = share_df['force'].str.replace(' Constabulary','').str.replace(' Police','').str.replace(' Service','')
    y = np.arange(len(share_df))
    w = 0.38
    axes[2].barh(y - w/2, share_df['demand_share_pct'],     w, label='Prophet',  color='#4CAF50', alpha=0.85)
    axes[2].barh(y + w/2, share_df['tfm_demand_share_pct'], w, label='TimesFM', color='#2196F3', alpha=0.85)
    axes[2].set_yticks(y)
    axes[2].set_yticklabels(share_short, fontsize=8)
    axes[2].set_xlabel('Demand share % of total')
    axes[2].set_title('Prophet vs TimesFM\nDemand Share % (top 20)', fontsize=10)
    axes[2].legend()

plt.tight_layout()
plt.savefig('../outputs/prophet_all_forces_budget_view.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Save outputs

In [ ]:
os.makedirs('../outputs', exist_ok=True)
forecast_df.to_parquet('../outputs/prophet_all_forces_2026_forecasts.parquet', index=False)
forecast_df[['force','month','forecast']].assign(
    month=lambda d: d['month'].dt.strftime('%Y-%m'),
    forecast=lambda d: d['forecast'].clip(lower=0).round().astype(int)
).to_csv('../outputs/prophet_all_forces_2026_forecasts.csv', index=False)
eval_df.to_csv('../outputs/prophet_all_forces_q1_eval.csv', index=False)
budget_view.to_csv('../outputs/prophet_all_forces_budget_view.csv', index=False)
print('Saved:')
print('  prophet_all_forces_2026_forecasts.parquet — full Jan–Dec 2026 forecasts per force')
print('  prophet_all_forces_2026_forecasts.csv     — same, CSV format')
print('  prophet_all_forces_q1_eval.csv            — Jan–Mar evaluation metrics')
print('  prophet_all_forces_budget_view.csv        — budget allocation summary table (full year)')